<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkML311Coursera747-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Deep Learning and Reinforcement Learning**

## Module 2 - BackPropagation Training and Keras
### Pengenalan Keras untuk Membangun Jaringan Saraf Tiruan

Estimasi waktu yang dibutuhkan: **45** menit

Dalam lab ini, kita akan menggunakan jaringan saraf tiruan untuk memprediksi diabetes menggunakan **Pima Diabetes Dataset**. Kita akan mulai dengan melatih model *Random Forest* untuk mendapatkan baseline performa. Kemudian, kita akan menggunakan pustaka **Keras** untuk membangun dan melatih jaringan saraf dengan cepat, serta membandingkan performanya. Kita juga akan melihat bagaimana struktur jaringan yang berbeda mempengaruhi performa, waktu pelatihan, dan tingkat *overfitting* atau *underfitting*.

## Daftar Isi

<ol>
    <li><a href="#Objectives">Tujuan Pembelajaran</a></li>
    <li>
        <a href="#Setup">Persiapan (Setup)</a>
        <ol>
            <li><a href="#Installing-Required-Libraries">Menginstal Pustaka yang Diperlukan</a></li>
            <li><a href="#Importing-Required-Libraries">Mengimpor Pustaka yang Diperlukan</a></li>
        </ol>       
    </li>
    <li><a href="#Background">Latar Belakang: UCI Pima Diabetes Dataset</a></li>
    <li><a href="#Exercise-1">Latihan 1: Mendapatkan Baseline dengan Random Forest</a></li>
    <li><a href="#Build-Neural-Network">Membangun Jaringan Saraf dengan Satu Lapisan Tersembunyi</a></li>
    <li><a href="#Exercise-2">Latihan 2: Eksperimen dengan Struktur Jaringan Berbeda</a></li>
</ol>


## Tujuan Pembelajaran

Setelah menyelesaikan lab ini, Anda akan mampu:

*   Menggunakan pustaka Keras untuk membangun model *Sequential*.
*   Melakukan normalisasi data untuk meningkatkan stabilitas pelatihan jaringan saraf.
*   Membandingkan performa Jaringan Saraf Tiruan dengan model klasifikasi tradisional (*Random Forest*).
*   Menganalisis kurva *loss* dan *accuracy* untuk mendeteksi *overfitting*.

## Persiapan (Setup)


Untuk lab ini, kita akan menggunakan beberapa pustaka berikut:

*   [`numpy`](https://numpy.org/) untuk operasi matematika.
*   [`pandas`](https://pandas.pydata.org/) untuk manipulasi data.
*   [`matplotlib`](https://matplotlib.org/) & [`seaborn`](https://seaborn.pydata.org/) untuk visualisasi.
*   [`scikit-learn`](https://scikit-learn.org/) untuk preprocessing, pembagian data, dan evaluasi.
*   [`tensorflow/keras`](https://www.tensorflow.org/guide/keras) untuk membangun jaringan saraf tiruan.

### Menginstal Pustaka yang Diperlukan

Hapus tanda `#` di bawah ini jika Anda menjalankan notebook di lingkungan lokal.

In [ ]:
# !pip install numpy pandas matplotlib seaborn scikit-learn tensorflow skillsnetwork

### Mengimpor Pustaka yang Diperlukan

Kita disarankan untuk mengimpor semua pustaka di satu tempat:

In [ ]:
import warnings
import skillsnetwork
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve, accuracy_score
from sklearn.ensemble import RandomForestClassifier

## Import Keras objects for Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

## Latar Belakang: UCI Pima Diabetes Dataset

Dataset ini berasal dari *National Institute of Diabetes and Digestive and Kidney Diseases*. Tujuannya adalah untuk memprediksi secara diagnostik apakah seorang pasien menderita diabetes, berdasarkan pengukuran diagnostik tertentu yang disertakan dalam dataset.

### Atribut (Semua bernilai numerik):
1.  **times_pregnant**: Berapa kali hamil.
2.  **glucose_tolerance_test**: Konsentrasi glukosa plasma 2 jam dalam tes toleransi glukosa oral.
3.  **blood_pressure**: Tekanan darah diastolik (mm Hg).
4.  **skin_thickness**: Ketebalan lipatan kulit trisep (mm).
5.  **insulin**: Serum insulin 2 jam (mu U/ml).
6.  **bmi**: Indeks massa tubuh (berat dalam kg/(tinggi dalam m)^2).
7.  **pedigree_function**: Riwayat diabetes dalam keluarga.
8.  **age**: Usia (tahun).
9.  **has_diabetes**: Variabel kelas (0 atau 1).

In [ ]:
## Load in the data set 
await skillsnetwork.prepare("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML311-Coursera/labs/Module2/L2/diabetes.csv", overwrite=True)

names = ["times_pregnant", "glucose_tolerance_test", "blood_pressure", "skin_thickness", "insulin", 
         "bmi", "pedigree_function", "age", "has_diabetes"]
diabetes_df = pd.read_csv('./diabetes.csv', names=names, header=0)

In [ ]:
# Take a peek at the data -- if there are lots of "NaN" you may have internet connectivity issues
print(diabetes_df.shape)
diabetes_df.sample(5)

In [ ]:
X = diabetes_df.iloc[:, :-1].values
y = diabetes_df["has_diabetes"].values

In [ ]:
# Split the data to Train, and Test (75%, 25%)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=11111)

In [ ]:
np.mean(y), np.mean(1-y)

## Latihan 1: Mendapatkan Baseline dengan Random Forest

Di atas, kita melihat bahwa sekitar 35% pasien dalam dataset ini menderita diabetes, sementara 65% tidak. Ini berarti kita bisa mendapatkan akurasi 65% tanpa model apa pun - cukup nyatakan bahwa tidak ada yang menderita diabetes.

Untuk memulai, mari kita dapatkan baseline performa menggunakan pengklasifikasi Random Forest:
1. Latih model Random Forest dengan 200 pohon pada data pelatihan.
2. Hitung akurasi dan skor ROC-AUC dari prediksi tersebut.

In [ ]:
### BEGIN SOLUTION
## Train the RF Model
rf_model = RandomForestClassifier(n_estimators=200)
rf_model.fit(X_train, y_train)

In [ ]:
# Make predictions on the test set - both "hard" predictions, and the scores (percent of trees voting yes)
y_pred_class_rf = rf_model.predict(X_test)
y_pred_prob_rf = rf_model.predict_proba(X_test)


print('accuracy is {:.3f}'.format(accuracy_score(y_test,y_pred_class_rf)))
print('roc-auc is {:.3f}'.format(roc_auc_score(y_test,y_pred_prob_rf[:,1])))

In [ ]:
def plot_roc(y_test, y_pred, model_name):
    fpr, tpr, thr = roc_curve(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.plot(fpr, tpr, 'k-')
    ax.plot([0, 1], [0, 1], 'k--', linewidth=.5)  # roc curve for random model
    ax.grid(True)
    ax.set(title='ROC Curve for {} on PIMA diabetes problem'.format(model_name),
           xlim=[-0.01, 1.01], ylim=[-0.01, 1.01])
plot_roc(y_test, y_pred_prob_rf[:, 1], 'RF')
### END SOLUTION

## Membangun Jaringan Saraf dengan Satu Lapisan Tersembunyi

Kita akan menggunakan model **Sequential** untuk membangun jaringan saraf dengan cepat. Jaringan pertama kita akan memiliki satu lapisan tersembunyi dengan 12 node. Karena kita memiliki 8 variabel input, kita mengatur `input_shape` ke 8.

In [ ]:
## First let's normalize the data
## This aids the training of neural nets by providing numerical stability
## Random Forest does not need this as it finds a split only, as opposed to performing matrix multiplications

normalizer = StandardScaler()
X_train_norm = normalizer.fit_transform(X_train)
X_test_norm = normalizer.transform(X_test)

In [ ]:
# Define the Model 
# Input size is 8-dimensional
# 1 hidden layer, 12 hidden nodes, sigmoid activation
# Final layer has just one node with a sigmoid activation (standard for binary classification)

model_1 = Sequential()
model_1.add(Dense(12,input_shape = (8,),activation = 'sigmoid'))
model_1.add(Dense(1,activation='sigmoid'))

In [ ]:
#  This is a nice tool to view the model you have created and count the parameters

model_1.summary()

### Comprehension question:
Why do we have 121 parameters?  Does that make sense?

Let's fit our model for 200 epochs.


In [ ]:
# Fit(Train) the Model

# Compile the model with Optimizer, Loss Function and Metrics
# Roc-Auc is not available in Keras as an off the shelf metric yet, so we will skip it here.

model_1.compile(SGD(lr = .003), "binary_crossentropy", metrics=["accuracy"])
run_hist_1 = model_1.fit(X_train_norm, y_train, validation_data=(X_test_norm, y_test), epochs=200)
# the fit function returns the run history. 
# It is very convenient, as it contains information about the model fit, iterations etc.

In [ ]:
## Like we did for the Random Forest, we generate two kinds of predictions
#  One is a hard decision, the other is a probabilitistic score.

y_pred_prob_nn_1 = model_1.predict(X_test_norm)
y_pred_class_nn_1 = (y_pred_prob_nn_1 > 0.5).astype(int)

In [ ]:
# Let's check out the outputs to get a feel for how keras apis work.
y_pred_class_nn_1[:10]

In [ ]:
y_pred_prob_nn_1[:10]

In [ ]:
# Print model performance and plot the roc curve
print('accuracy is {:.3f}'.format(accuracy_score(y_test,y_pred_class_nn_1)))
print('roc-auc is {:.3f}'.format(roc_auc_score(y_test,y_pred_prob_nn_1)))

plot_roc(y_test, y_pred_prob_nn_1, 'NN')

There may be some variation in exact numbers due to randomness, but you should get results in the same ballpark as the Random Forest - between 75% and 85% accuracy, between .8 and .9 for AUC.


Mari kita periksa objek `run_hist_1` yang telah dibuat, khususnya atribut `history`-nya.


In [ ]:
run_hist_1.history.keys()

Mari kita buat grafik *training loss* dan *validation loss* di berbagai epoch.


In [ ]:
fig, ax = plt.subplots()
ax.plot(run_hist_1.history["loss"],'r', marker='.', label="Train Loss")
ax.plot(run_hist_1.history["val_loss"],'b', marker='.', label="Validation Loss")
ax.legend()

Terlihat bahwa *loss* masih menurun. Mari kita latih model selama 1000 epoch tambahan.


In [ ]:
## Note that when we call "fit" again, it picks up where it left off
run_hist_1b = model_1.fit(X_train_norm, y_train, validation_data=(X_test_norm, y_test), epochs=1000)

In [ ]:
n = len(run_hist_1.history["loss"])
m = len(run_hist_1b.history['loss'])
fig, ax = plt.subplots(figsize=(16, 8))

ax.plot(range(n), run_hist_1.history["loss"],'r', marker='.', label="Train Loss - Run 1")
ax.plot(range(n, n+m), run_hist_1b.history["loss"], 'hotpink', marker='.', label="Train Loss - Run 2")

ax.plot(range(n), run_hist_1.history["val_loss"],'b', marker='.', label="Validation Loss - Run 1")
ax.plot(range(n, n+m), run_hist_1b.history["val_loss"], 'LightSkyBlue', marker='.',  label="Validation Loss - Run 2")

ax.legend()

Terlihat bahwa *loss* masih menurun. Mari kita latih model selama 1000 epoch tambahan.


## Latihan 2: Eksperimen dengan Struktur Jaringan Berbeda

Untuk latihan ini, lakukan hal berikut pada sel di bawah ini:
- Bangun model dengan dua lapisan tersembunyi, masing-masing dengan 6 node.
- Gunakan fungsi aktivasi "relu" untuk lapisan tersembunyi, dan "sigmoid" untuk lapisan akhir.
- Gunakan *learning rate* 0,003 dan latih selama 1500 epoch.
- Buat grafik trajektori fungsi *loss* dan akurasi pada set data *train* dan *test*.
- Plot kurva ROC untuk hasil prediksi.

Cobalah bereksperimen dengan *learning rate*, jumlah epoch, dan struktur jaringan yang berbeda untuk melihat pengaruhnya!

In [ ]:
### BEGIN SOLUTION
model_2 = Sequential()
model_2.add(Dense(6, input_shape=(8,), activation="relu"))
model_2.add(Dense(6,  activation="relu"))
model_2.add(Dense(1, activation="sigmoid"))

model_2.compile(SGD(lr = .003), "binary_crossentropy", metrics=["accuracy"])
run_hist_2 = model_2.fit(X_train_norm, y_train, validation_data=(X_test_norm, y_test), epochs=1500)

In [ ]:
run_hist_2.history.keys()

In [ ]:
n = len(run_hist_2.history["loss"])

fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 2, 1)
ax.plot(range(n), (run_hist_2.history["loss"]),'r.', label="Train Loss")
ax.plot(range(n), (run_hist_2.history["val_loss"]),'b.', label="Validation Loss")
ax.legend()
ax.set_title('Loss over iterations')

ax = fig.add_subplot(1, 2, 2)
ax.plot(range(n), (run_hist_2.history["acc"]),'r.', label="Train Acc")
ax.plot(range(n), (run_hist_2.history["val_acc"]),'b.', label="Validation Acc")
ax.legend(loc='lower right')
ax.set_title('Accuracy over iterations')

In [ ]:
y_pred_class_nn_2 = model_2.predict_classes(X_test_norm)
y_pred_prob_nn_2 = model_2.predict(X_test_norm)
print('')
print('accuracy is {:.3f}'.format(accuracy_score(y_test,y_pred_class_nn_2)))
print('roc-auc is {:.3f}'.format(roc_auc_score(y_test,y_pred_prob_nn_2)))

plot_roc(y_test, y_pred_prob_nn_2, 'NN-2')
### END SOLUTION

### Kesimpulan

Dalam lab ini, kita telah mempelajari:
1. Cara menggunakan **Keras** untuk membangun model jaringan saraf tiruan secara modular.
2. Pentingnya normalisasi data sebelum melatih jaringan saraf.
3. Cara mengevaluasi performa model menggunakan akurasi dan **ROC-AUC**.
4. Cara memantau proses pelatihan melalui kurva *loss* dan akurasi untuk mendeteksi *overfitting*.

## Log Perubahan

| Tanggal (YYYY-MM-DD) | Versi | Diubah Oleh | Deskripsi Perubahan |
| ----------------- | ------- | ----------- | ------------------ |
| 2020-07-15        | 0.1     | Steve Hord  | Created Lab       |


Copyright © 2022 IBM Corporation. All rights reserved.